# 03 - Keyword Extraction & Thematic Analysis
## Amazon Workforce Analytics Project

**Objective:** Extract keywords, bigrams, and themes from employee feedback using NLTK.

**Tools:** NLTK, WordCloud, Pandas, Matplotlib

**Input:** Cleaned text data  
**Output:** Top keywords, bigrams, word clouds, and theme assignments

---


## 1. Setup & Install Libraries

In [ ]:
!pip install nltk wordcloud -q

import nltk
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from nltk.util import ngrams
from nltk.corpus import stopwords
from wordcloud import WordCloud

print("✅ Libraries imported!")

## 2. Load Sentiment Data

In [ ]:
# Load data with sentiment scores
glassdoor_df = pd.read_csv('glassdoor_sentiment.csv')
youtube_df = pd.read_csv('youtube_sentiment.csv')

print(f"Glassdoor: {len(glassdoor_df)} rows")
print(f"YouTube: {len(youtube_df)} rows")

## 3. Define Domain-Specific Stopwords

Standard stopwords miss domain-specific common words. We add Amazon/warehouse-related terms that don't provide analytical value.


In [ ]:
# Get standard English stopwords
stop_words = set(stopwords.words('english'))

# Add domain-specific stopwords
domain_stopwords = {
    'amazon', 'work', 'working', 'job', 'like', 'just', 'really', 
    'dont', 'know', 'going', 'get', 'got', 'go', 'would', 'could',
    'one', 'also', 'much', 'even', 'make', 'way', 'thing', 'things',
    'lot', 'come', 'say', 'said', 'youre', 'thats', 'theyre', 'ive',
    'im', 'didnt', 'doesnt', 'cant', 'wont', 'isnt', 'arent', 'wasnt'
}

# Combine stopwords
all_stopwords = stop_words.union(domain_stopwords)
print(f"Total stopwords: {len(all_stopwords)}")

## 4. Extract Keywords (Unigrams) - Glassdoor

In [ ]:
def extract_keywords(text_series, stopwords, top_n=20):
    """Extract top keywords from a series of text"""
    # Combine all text
    all_text = ' '.join(text_series.dropna().astype(str))
    
    # Tokenize and filter
    words = all_text.lower().split()
    words = [w for w in words if w not in stopwords and len(w) > 2]
    
    # Count frequencies
    word_counts = Counter(words)
    return word_counts.most_common(top_n)

# Extract Glassdoor keywords
glassdoor_keywords = extract_keywords(glassdoor_df['cleaned_text'], all_stopwords)

print("=== Top 20 Glassdoor Keywords ===")
for word, count in glassdoor_keywords:
    print(f"  {word}: {count}")

## 5. Extract Bigrams (Two-Word Phrases)

Bigrams reveal more context than single words. "bathroom break" tells more than "bathroom" or "break" alone.


In [ ]:
def extract_bigrams(text_series, stopwords, top_n=15):
    """Extract top bigrams from a series of text"""
    all_bigrams = []
    
    for text in text_series.dropna():
        words = str(text).lower().split()
        words = [w for w in words if w not in stopwords and len(w) > 2]
        bigram_list = list(ngrams(words, 2))
        all_bigrams.extend(bigram_list)
    
    bigram_counts = Counter(all_bigrams)
    return bigram_counts.most_common(top_n)

# Extract Glassdoor bigrams
glassdoor_bigrams = extract_bigrams(glassdoor_df['cleaned_text'], all_stopwords)

print("=== Top 15 Glassdoor Bigrams ===")
for bigram, count in glassdoor_bigrams:
    print(f"  {' '.join(bigram)}: {count}")

## 6. Extract Keywords & Bigrams - YouTube

In [ ]:
# YouTube keywords
youtube_keywords = extract_keywords(youtube_df['transcript_cleaned'], all_stopwords)

print("=== Top 20 YouTube Keywords ===")
for word, count in youtube_keywords:
    print(f"  {word}: {count}")

In [ ]:
# YouTube bigrams
youtube_bigrams = extract_bigrams(youtube_df['transcript_cleaned'], all_stopwords)

print("=== Top 15 YouTube Bigrams ===")
for bigram, count in youtube_bigrams:
    print(f"  {' '.join(bigram)}: {count}")

## 7. Create Word Clouds

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Glassdoor word cloud
glassdoor_text = ' '.join(glassdoor_df['cleaned_text'].dropna().astype(str))
wc_glassdoor = WordCloud(width=800, height=400, background_color='white',
                         stopwords=all_stopwords, colormap='Blues').generate(glassdoor_text)
axes[0].imshow(wc_glassdoor, interpolation='bilinear')
axes[0].axis('off')
axes[0].set_title('Glassdoor Reviews - Word Cloud', fontsize=14, fontweight='bold')

# YouTube word cloud
youtube_text = ' '.join(youtube_df['transcript_cleaned'].dropna().astype(str))
wc_youtube = WordCloud(width=800, height=400, background_color='white',
                       stopwords=all_stopwords, colormap='Oranges').generate(youtube_text)
axes[1].imshow(wc_youtube, interpolation='bilinear')
axes[1].axis('off')
axes[1].set_title('YouTube Transcripts - Word Cloud', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('wordclouds_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Saved: wordclouds_comparison.png")

## 8. Assign Themes Based on Keywords

We identify three main friction themes based on keyword patterns:
1. **Physical Labor Intensity** - fatigue, physical, standing, walking, lifting
2. **Shift Length & Break Policy** - hours, shift, break, overtime, schedule
3. **Inconsistent Management** - management, manager, supervisor, communication


In [ ]:
def assign_theme(text):
    """Assign friction theme based on keywords"""
    text = str(text).lower()
    
    # Theme keyword patterns
    physical_keywords = ['physical', 'fatigue', 'tired', 'standing', 'walking', 
                         'lifting', 'body', 'feet', 'back', 'pain', 'exhausting']
    shift_keywords = ['hours', 'shift', 'break', 'overtime', 'schedule', 'time',
                      '10hour', '12hour', 'mandatory', 'pto', 'vacation']
    management_keywords = ['management', 'manager', 'supervisor', 'leadership',
                          'communication', 'communicate', 'favoritism', 'hr']
    
    # Count matches
    physical_count = sum(1 for kw in physical_keywords if kw in text)
    shift_count = sum(1 for kw in shift_keywords if kw in text)
    management_count = sum(1 for kw in management_keywords if kw in text)
    
    # Assign theme based on highest count
    counts = {
        'Physical Labor Intensity': physical_count,
        'Shift Length & Break Policy': shift_count,
        'Inconsistent Management': management_count
    }
    
    max_theme = max(counts, key=counts.get)
    if counts[max_theme] > 0:
        return max_theme
    return 'Other'

# Apply to Glassdoor
glassdoor_df['Theme'] = glassdoor_df['cleaned_text'].apply(assign_theme)

print("=== Theme Distribution ===")
print(glassdoor_df['Theme'].value_counts())

## 9. Analyze Sentiment by Theme

In [ ]:
# Calculate negative sentiment percentage by theme
theme_sentiment = glassdoor_df.groupby('Theme').agg({
    'sentiment_label': lambda x: (x == 'negative').mean() * 100,
    'cleaned_text': 'count'
}).rename(columns={'sentiment_label': 'Negative %', 'cleaned_text': 'Count'})

theme_sentiment = theme_sentiment.sort_values('Negative %', ascending=False)

print("=== Sentiment by Theme ===")
print(theme_sentiment.round(1))
print(f"\n🔍 Insight: '{theme_sentiment.index[0]}' has the highest negative sentiment!")

## 10. Save Results

In [ ]:
# Save Glassdoor with themes
glassdoor_df.to_csv('glassdoor_with_themes.csv', index=False)
print("✅ Saved: glassdoor_with_themes.csv")

# Save keyword analysis
keyword_df = pd.DataFrame({
    'Glassdoor_Keyword': [w for w, c in glassdoor_keywords],
    'Glassdoor_Count': [c for w, c in glassdoor_keywords],
    'YouTube_Keyword': [w for w, c in youtube_keywords],
    'YouTube_Count': [c for w, c in youtube_keywords]
})
keyword_df.to_csv('keyword_comparison.csv', index=False)
print("✅ Saved: keyword_comparison.csv")

---
## Key Findings

**Top YouTube Bigrams (NLTK revealed what TextBlob missed):**
- "bathroom break" - Major operational concern
- "conveyor belt" - Physical work specifics
- "10hour shift" - Work hour complaints

**Theme Analysis:**
- Management theme had HIGHEST negative sentiment (43%)
- Physical labor had MORE mentions but LOWER negativity
- **Insight:** Volume ≠ Severity

**Next:** `04_segmentation_analysis.ipynb` - Segment workforce and prioritize interventions
